In [23]:
#Imports
import pandas as pd

In [24]:
#Load datasets
sold = pd.read_csv("soldRates.csv")
listings = pd.read_csv("listingsRates.csv")


/var/folders/l4/hlg0lrhx2_79czgk4q732ylw0000gq/T/ipykernel_54591/1734972999.py:2: DtypeWarning: Columns (0,1,7,51,63,64,65) have mixed types. Specify dtype option on import or set low_memory=False.
  sold = pd.read_csv("soldRates.csv")
/var/folders/l4/hlg0lrhx2_79czgk4q732ylw0000gq/T/ipykernel_54591/1734972999.py:3: DtypeWarning: Columns (2,69) have mixed types. Specify dtype option on import or set low_memory=False.
  listings = pd.read_csv("listingsRates.csv")


In [25]:
#Before Row counts
print(f" sold:{len(sold):,} rows")
print(f"  istings:{len(listings):,} rows")

 sold:381,848 rows
  istings:515,195 rows


Convert date fields to datetime format

In [26]:
#Convert date fields to datetime format (CloseDate, PurchaseContractDate, ListingContractDate, ContractStatusChangeDate)
date_cols = ["CloseDate", "PurchaseContractDate", "ListingContractDate","ContractStatusChangeDate"]
for col in date_cols:
    if col in sold.columns:
        sold[col] = pd.to_datetime(sold[col], errors='coerce')
    if col in listings.columns:
        listings[col] = pd.to_datetime(listings[col], errors='coerce')

Remove unnecessary or redundant columns

In [27]:
#Remove unnecessary or redundant columns
drop_cols = [
    "ListingKeyNumeric", "ListingId",
    "ListAgentFirstName", "ListAgentLastName",
    "CoListAgentFirstName", "CoListAgentLastName",
    "BuyerAgentFirstName", "BuyerAgentLastName",
    "latfilled", "lonfilled",
    "ListAgentEmail", "BuyerAgentMlsId",
    "BuyerAgentAOR", "ListAgentAOR", "BuyerOfficeAOR",
    "BuyerOfficeName", "CoListOfficeName", "ListOfficeName"
]

sold = sold.drop(columns=[c for c in drop_cols if c in sold.columns])
listings = listings.drop(columns=[c for c in drop_cols if c in listings.columns])

Ensure numeric fields are properly typed

In [28]:
#Finding which columns have numeric values in the sold dataset
numeric_cols = sold.select_dtypes(include=['number']).columns
print(sold[numeric_cols].dtypes)

OriginalListPrice          float64
ListingKey                   int64
ClosePrice                 float64
Latitude                   float64
Longitude                  float64
LivingArea                 float64
ListPrice                  float64
DaysOnMarket                 int64
ParkingTotal               float64
LotSizeAcres               float64
YearBuilt                  float64
StreetNumberNumeric        float64
BathroomsTotalInteger      float64
BedroomsTotal              float64
Stories                    float64
LotSizeArea                float64
MainLevelBedrooms          float64
GarageSpaces               float64
AssociationFee             float64
LotSizeSquareFeet          float64
BuyerAgencyCompensation    float64
rate_30yr_fixed            float64
dtype: object


In [29]:
#Ensure numeric fields are properly typed for sold csv
for col in numeric_cols:
    sold[col] = pd.to_numeric(sold[col], errors='coerce')

In [30]:
#Finding which columns have numeric values in the lisitings dataset
numeric_cols = listings.select_dtypes(include=['number']).columns
print(listings[numeric_cols].dtypes)

OriginalListPrice          float64
ListingKey                   int64
ClosePrice                 float64
Latitude                   float64
Longitude                  float64
LivingArea                 float64
ListPrice                  float64
DaysOnMarket                 int64
ParkingTotal               float64
LotSizeAcres               float64
YearBuilt                  float64
DaysOnMarket.1               int64
StreetNumberNumeric        float64
LivingArea.1               float64
BathroomsTotalInteger      float64
BedroomsTotal              float64
Longitude.1                float64
Latitude.1                 float64
ListPrice.1                float64
Stories                    float64
LotSizeArea                float64
MainLevelBedrooms          float64
GarageSpaces               float64
AssociationFee             float64
LotSizeSquareFeet          float64
BuyerAgencyCompensation    float64
rate_30yr_fixed            float64
dtype: object


In [31]:
#Ensure numeric fields are properly typed for listings csv
for col in numeric_cols:
    listings[col] = pd.to_numeric(listings[col], errors='coerce')

Remove or flag invalid numeric values

In [32]:
#Remove or flag invalid numeric values: ClosePrice <= 0, LivingArea <= 0, DaysOnMarket < 0, negative Bedrooms or Bathrooms for Sold CSV
sold["invalid_price_flag"] = sold["ClosePrice"] <= 0
sold["invalid_area_flag"] = sold["LivingArea"] <= 0
sold["invalid_dom_flag"] = sold["DaysOnMarket"] < 0
sold["invalid_bed_flag"] = sold["BedroomsTotal"] < 0
sold["invalid_bath_flag"] = sold["BathroomsTotalInteger"] < 0

In [33]:
#Remove or flag invalid numeric values: ClosePrice <= 0, LivingArea <= 0, DaysOnMarket < 0, negative Bedrooms or Bathrooms for Listings CSV
listings["invalid_price_flag"] = listings["ClosePrice"] <= 0
listings["invalid_area_flag"] = listings["LivingArea"] <= 0
listings["invalid_dom_flag"] = listings["DaysOnMarket"] < 0
listings["invalid_bed_flag"] = listings["BedroomsTotal"] < 0
listings["invalid_bath_flag"] = listings["BathroomsTotalInteger"] < 0

Date Consistency Checks

In [ ]:
#ListingContractDate should precede PurchaseContractDate, which should precede CloseDate for sold csv.
sold["listing_after_close_flag"] = (sold["ListingContractDate"].notna() & sold["CloseDate"].notna() & (sold["ListingContractDate"] > sold["CloseDate"]))

sold["purchase_after_close_flag"] = (sold["PurchaseContractDate"].notna() & sold["CloseDate"].notna() & (sold["PurchaseContractDate"] > sold["CloseDate"]))

sold["negative_timeline_flag"] = (sold["ListingContractDate"].notna() & sold["PurchaseContractDate"].notna() & sold["CloseDate"].notna() &( (sold["PurchaseContractDate"] < sold["ListingContractDate"]) |(sold["CloseDate"] < sold["PurchaseContractDate"])))

In [ ]:
#ListingContractDate should precede PurchaseContractDate, which should precede CloseDate for lisitings csv.
listings["listing_after_close_flag"] = (listings["ListingContractDate"].notna() & listings["CloseDate"].notna() & (listings["ListingContractDate"] > listings["CloseDate"]))

listings["purchase_after_close_flag"] = (listings["PurchaseContractDate"].notna() & listings["CloseDate"].notna() & (listings["PurchaseContractDate"] > listings["CloseDate"]))

listings["negative_timeline_flag"] = (listings["ListingContractDate"].notna() & listings["PurchaseContractDate"].notna() & listings["CloseDate"].notna() & ( (listings["PurchaseContractDate"] < listings["ListingContractDate"]) | (listings["CloseDate"] < listings["PurchaseContractDate"])))

Geographic Data Checks

In [ ]:
#Flag records with missing coordinates (Latitude or Longitude is null)
sold["missing_coords_flag"] = (sold["Latitude"].isna() | sold["Longitude"].isna())

listings["missing_coords_flag"] = (listings["Latitude"].isna() | listings["Longitude"].isna())

In [ ]:
#Flag Latitude = 0 or Longitude = 0 (sentinel null values)
sold["zero_coords_flag"] = ((sold["Latitude"] == 0) | (sold["Longitude"] == 0))

listings["zero_coords_flag"] = ((listings["Latitude"] == 0) | (listings["Longitude"] == 0))

In [38]:
#Flag Longitude > 0 errors (California coordinates should be negative)
sold["positive_longitude_flag"] = sold["Longitude"] > 0
listings["positive_longitude_flag"] = listings["Longitude"] > 0

In [ ]:
#Flag out-of-state or implausible coordinates
sold["out_of_state_flag"] = ((sold["Latitude"] < 32) | (sold["Latitude"] > 42) | (sold["Longitude"] < -125) | (sold["Longitude"] > -114))

listings["out_of_state_flag"] = ((listings["Latitude"] < 32) | (listings["Latitude"] > 42) |(listings["Longitude"] < -125) | (listings["Longitude"] > -114))

In [41]:
#After Row counts
print(f" sold:{len(sold):} rows")
print(f" istings:{len(listings):} rows")

 sold:381848 rows
 istings:515195 rows
